In [1]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import numpy as np


DATA_FOLDER = "./"

df = pd.read_pickle(DATA_FOLDER + "df_fe_epic_light_best_customers_25c.pickle")

product_ids = pd.read_csv(DATA_FOLDER + "product_id_apredecir201912.txt", sep="\t")[
    "product_id"
].tolist()
#df = df[df["product_id"].isin(product_ids)]
df.drop(columns=["periodo"], inplace=True, errors="ignore")


/home/fede/.venvs/labo3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
numeric_df = df.select_dtypes(include=[np.number])
total_infs = np.isinf(numeric_df.values).sum()
print(f"Total infs: {total_infs}")
df[numeric_df.columns] = df[numeric_df.columns].replace([np.inf, -np.inf], np.nan)

Total infs: 0


In [3]:
df["serie_id"] = df["product_id"].astype(str) + "_" + df["customer_id"].astype(str)
#df['fecha'] = df['fecha'].apply(lambda x: x.to_timestamp('M'))  # último día del mes
unique_fechas = df["fecha"].unique()
# aplico el to_timestamp('M') a las fechas unicas y despues reemplazo en el df
unique_fechas_transform = [x.to_timestamp('M') for x in unique_fechas]
df['fecha'] = df['fecha'].replace(unique_fechas, unique_fechas_transform)
del unique_fechas, unique_fechas_transform

/tmp/ipykernel_172997/105179847.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["serie_id"] = df["product_id"].astype(str) + "_" + df["customer_id"].astype(str)
/tmp/ipykernel_172997/105179847.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['fecha'] = df['fecha'].replace(unique_fechas, unique_fechas_transform)


In [4]:
TEST_DATE = 33

def get_indexes_tabular(df, test_date=TEST_DATE):
    test_index = df.index[df['date_id'] == test_date]
    train_index = df.index[df['date_id'] <= test_date-3]
    val_index = df.index[df['date_id'] == test_date-2]

    train_scaler_index = df.index[df['date_id'] <= test_date]
    return test_index, train_index, val_index, train_scaler_index

test_index, train_index, val_index, train_scaler_index = get_indexes_tabular(df)


In [5]:
# pruebo transformando el target (ahora si es relevante en tabular)
import re
transformations = {
    "tn": [
        r"tn$",
        r"cust_request_qty_per_tn$",
        r"tn_lag_*",
        r"tn_rolling_mean_*",
        r"tn_rolling_max_*",
        r"tn_rolling_min_*",
        r"tn_.*_vendidas$",
        r"tn_agg*",
    ]
    + [r"stock_final$"]
    + [r"cust_request_tn_minus_tn$"]
    + [r"tn_diff_*"],
    "cust_request_qty": [
        r"cust_request_qty$",
        r"cust_request_qty_lag_*",
        r"cust_request_qty_rolling_mean_*",
        r"cust_request_qty_rolling_max_*",
        r"cust_request_qty_rolling_min_*",
        r"cust_request_qty_.*_vendidas$",
        r"cust_request_qty_agg*",
    ]
    + [r"cust_request_qty_diff_*"],
}

def scale_df(df, transformations, train_scaler_index):
    numeric_columns = df.select_dtypes(include=["float64", "float32", "int32", "int64"]).columns
    print(numeric_columns)
    df_scaled = df  # no hago copy intencionalmente
    train_scaler_df = df_scaled.loc[train_scaler_index]


    group_stats = train_scaler_df.groupby(["customer_id", "product_id"])[
        list(transformations.keys())
    ].agg(["mean", "std"]).reset_index()
    #print(prod_stats.head())
    group_stats.columns = [
        f"{col[0]}_{col[1]}" if col[1] else col[0] for col in group_stats.columns
    ]  # aplanar el multiindex de las columnas
    # replace infs with NaN
    group_stats = group_stats.replace([np.inf, -np.inf], np.nan)
    group_stats = group_stats.fillna(0)  # reemplazar NaN por
    print(group_stats.head())

    # Mergear las stats al df original
    df_scaled = df_scaled.merge(
        group_stats, on=["product_id", "customer_id"], how="left"
    )
    df_scaled = df_scaled.set_index(df.index)

    scaled_cols = {}
    for trainer, regex_cols in transformations.items():
        for col in regex_cols:
            # Usar regex para seleccionar las columnas que coinciden
            # chequear si la columna es un regex
            matching_cols = [c for c in numeric_columns if re.match(col, c)]
            if not matching_cols:
                continue  # Si no hay columnas que coincidan, saltar

            # Calcular la media y desviación estándar para cada
            print(f"Processing trainer: {trainer} with columns: {matching_cols}")
            # Escalar las columnas
            for col in matching_cols:
                scaled_cols[col + "_scaled"] = (df_scaled[col]) / df_scaled[
                    trainer + "_std"
                ]
                scaled_cols[col + "_scaled"].replace([np.inf, -np.inf], np.nan, inplace=True)
                scaled_cols[col + "_scaled"] = scaled_cols[col + "_scaled"].fillna(0)

    # Crear un DataFrame con todas las columnas escaladas
    scaled_df = pd.DataFrame(scaled_cols, index=df_scaled.index)

    # Concatenar de una sola vez
    df_scaled = pd.concat([df_scaled, scaled_df], axis=1)
    aux_cols = [col + "_mean" for col in list(transformations.keys())] + [
        col + "_std" for col in list(transformations.keys())
    ]
    df_scaled = df_scaled.drop(columns=aux_cols)
    return df_scaled, group_stats

df, group_stats = scale_df(df, transformations, train_scaler_index)
df

Index(['plan_precios_cuidados', 'cust_request_tn', 'tn', 'stock_final',
       'sku_size', 'coseno_fecha', 'seno_fecha', 'cust_request_tn_minus_tn',
       'tn_mult_cust_request_qty', 'tn_kama_indicator',
       ...
       'cust_request_qty_sku_size_vendidas',
       'cust_request_qty_sku_size_vendidas_div',
       'cust_request_qty_product_id_vendidas_div',
       'cust_request_qty_customer_id_vendidas',
       'cust_request_qty_customer_id_vendidas_div', 'tn_customer_vendidas',
       'tn_total_vendidas', 'tn_customer_weight', 'tn_product_vendidas',
       'tn_product_weight'],
      dtype='object', length=102)
   customer_id  product_id     tn_mean      tn_std  cust_request_qty_mean  \
0            0       20001  210.948578   70.598320             227.000000   
1            0       20002  203.586212   83.510956             219.970588   
2            0       20003  252.037399   84.934402             252.794118   
3            0       20004  332.865662  128.434845             260.7647

,product_id,customer_id,fecha,periodo_min_producto,periodo_max_producto,periodo_min_customer,periodo_max_customer,plan_precios_cuidados,cust_request_qty,cust_request_tn,...,cust_request_qty_rolling_min_12_scaled,cust_request_qty_cat3_vendidas_scaled,cust_request_qty_brand_vendidas_scaled,cust_request_qty_sku_size_vendidas_scaled,cust_request_qty_customer_id_vendidas_scaled,cust_request_qty_diff_1_scaled,cust_request_qty_diff_2_scaled,cust_request_qty_diff_3_scaled,cust_request_qty_diff_5_scaled,cust_request_qty_diff_11_scaled
0,20001,0,2017-01-31,NaT,NaT,NaT,NaT,NaN,229,139.028275,...,0.0,149.385159,67.892279,98.433655,1807.569647,0.0,0.0,0.0,0.0,0.0
1,20001,10001,2017-01-31,2017-01-01,2019-12-01,2017-01-01,2019-12-01,0.0,11,99.438606,...,0.0,711.619631,323.415519,468.904152,352.776971,0.0,0.0,0.0,0.0,0.0
2,20001,10002,2017-01-31,2017-01-01,2019-12-01,2017-01-01,2019-12-01,0.0,17,38.683010,...,0.0,568.409728,258.329758,374.539529,1030.791205,0.0,0.0,0.0,0.0,0.0
3,20001,10003,2017-01-31,2017-01-01,2019-12-01,2017-01-01,2019-12-01,0.0,17,143.494263,...,0.0,949.269423,431.422138,625.497604,959.471578,0.0,0.0,0.0,0.0,0.0
4,20001,10004,2017-01-31,2017-01-01,2019-12-01,2017-01-01,2019-12-01,0.0,9,184.729263,...,0.0,1587.124928,721.313478,1045.796709,1099.125676,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
819567,21276,10021,2019-12-31,2019-03-01,2019-12-01,2017-01-01,2019-12-01,0.0,0,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
819568,21276,10022,2019-12-31,2019-03-01,2019-12-01,2017-01-01,2019-12-01,0.0,0,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
819569,21276,10023,2019-12-31,2019-03-01,2019-12-01,2017-01-01,2019-12-01,0.0,0,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
819570,21276,10024,2019-12-31,2019-03-01,2019-12-01,2017-01-01,2019-12-01,0.0,0,0.000000,...,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0


In [6]:
df["target"] = df.groupby(['customer_id', 'product_id'])['tn_scaled'].shift(-2)
#df["sample_weight"] = df["tn"] + 1
# best features
#best_features = ['product_id', 'serie_id', 'tn_cat1_FOODS_div', 'tn_cat1_HC_div', 'tn_lag_1', 'ticket_promedio', 'tn_kama_indicator', 'tn_cat1_PC_div', 'brand', 'tn', 'cust_request_tn', 'cat3', 'sku_size', 'tn_brand_vendidas_div', 'tn_product_weight', 'tn_rolling_mean_12_lag_2', 'tn_rolling_max_6_scaled', 'tn_cat1_vendidas_scaled', 'tn_product_id_vendidas_scaled', 'tn_cat2_vendidas']
#best_features = ['product_id', 'tn_scaled', 'tn_customer_id_vendidas_scaled', 'stock_final_scaled', 'tn_product_id_vendidas_scaled', 'tn_total_vendidas_scaled', 'tn_cat2_vendidas_scaled', 'serie_id', 'tn_rolling_mean_12_scaled', 'tn_lag_1_scaled', 'tn_sku_size_vendidas_scaled', 'cust_request_qty_brand_vendidas_scaled', 'stock_final', 'cust_request_qty_scaled', 'cat2', 'tn_kama_indicator', 'cust_request_qty_product_id_vendidas_scaled', 'cust_request_qty_cat1_FOODS_div', 'tn_ulcer_index', 'cat3']
#df = df[best_features + ['target', 'product_id', 'fecha', 'serie_id']]
# drop duplicated columns
df = df.loc[:, ~df.columns.duplicated()]
train_df_tabular = df.loc[train_index].copy().dropna(subset=['target'])
val_df_tabular = df.loc[val_index].copy().dropna(subset=['target'])
train_df_tabular = pd.concat([train_df_tabular, val_df_tabular])
test_df_tabular = df.loc[test_index].copy()


In [7]:
# modelo tabular baseline
from autogluon.tabular import TabularDataset, TabularPredictor

train_data_tabular = TabularDataset(train_df_tabular)
test_data_tabular = TabularDataset(test_df_tabular)
import numpy as np
from sklearn.metrics import mean_absolute_error


    

predictor_tabular = TabularPredictor(label="target").fit(
    train_data_tabular, 
    #hyperparameters="toy",
    presets="medium_quality", 
    excluded_model_types=["RF", "XT"],
    #tuning_data=val_df_tabular,
    #time_limit=600,  # seconds
    #use_bag_holdout=True,
    
)
# exclude RF
y_pred_tabular = predictor_tabular.predict(test_data_tabular)
y_pred_tabular

No path specified. Models will be saved in: "AutogluonModels/ag-20250702_012344"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
Memory Avail:       8.68 GB / 15.32 GB (56.7%)
Disk Space Avail:   63.84 GB / 575.67 GB (11.1%)
Presets specified: ['medium_quality']
	Consider setting `time_limit` to ensure training finishes within an expected duration or experiment with a small portion of `train_data` to identify an ideal `presets` and `hyperparameters` configuration.
/home/fede/.venvs/labo3/lib/python3.12/site-packages/autogluon/common/utils/utils.py:97: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from 

[1000]	valid_set's rmse: 0.825763


KeyboardInterrupt: 

In [45]:
features = predictor_tabular.feature_importance(test_data_tabular)

These features in provided data are not utilized by the predictor and will be ignored: ['customer_id', 'tn_customer_vendidas', 'tn_product_vendidas', 'ticket_promedio_con_ceros', 'total_customer_con_ceros', 'tn_customer_vendidas_scaled', 'tn_product_vendidas_scaled']
Computing feature importance via permutation shuffling for 173 features using 972 rows with 5 shuffle sets...
	347.49s	= Expected runtime (69.5s per shuffle set)
	113.41s	= Actual runtime (Completed 5 of 5 shuffle sets)


In [48]:
# ordeno features por importancia y me quedo con las 20 mas importantes
features = features.sort_values(by='importance', ascending=False).head(20)
print(features.index.tolist())

['product_id', 'serie_id', 'tn_cat1_FOODS_div', 'tn_cat1_HC_div', 'tn_lag_1', 'ticket_promedio', 'tn_kama_indicator', 'tn_cat1_PC_div', 'brand', 'tn', 'cust_request_tn', 'cat3', 'sku_size', 'tn_brand_vendidas_div', 'tn_product_weight', 'tn_rolling_mean_12_lag_2', 'tn_rolling_max_6_scaled', 'tn_cat1_vendidas_scaled', 'tn_product_id_vendidas_scaled', 'tn_cat2_vendidas']


In [11]:
group_stats

,customer_id,product_id,tn_mean,tn_std,cust_request_qty_mean,cust_request_qty_std
0,0,20001,210.948578,70.598320,227.000000,54.188783
1,0,20002,203.586212,83.510956,219.970588,56.054889
2,0,20003,252.037399,84.934402,252.794118,51.868587
3,0,20004,332.865662,128.434845,260.764706,50.795525
4,0,20005,310.033936,109.603775,215.147059,51.807413
...,...,...,...,...,...,...
31949,10025,21295,0.000000,0.000000,0.000000,0.000000
31950,10025,21296,0.000000,0.000000,0.000000,0.000000
31951,10025,21297,0.000000,0.000000,0.000000,0.000000
31952,10025,21298,0.000000,0.000000,0.000000,0.000000


In [ ]:
predictions_tabular = test_df_tabular[["product_id", "customer_id", "target"]].copy()
predictions_tabular["prediction"] = y_pred_tabular
predictions_tabular = predictions_tabular.merge(group_stats[["product_id", "customer_id", "tn_std"]], on=["product_id", "customer_id"], how="left")
predictions_tabular["prediction"] = predictions_tabular["prediction"] * predictions_tabular["tn_std"]
predictions_tabular["target"] = predictions_tabular["target"] * predictions_tabular["tn_std"]


predictions_tabular = predictions_tabular.groupby(['product_id']).agg({
    'target': 'sum',
    'prediction': 'sum'
}).reset_index()
predictions_tabular = predictions_tabular[predictions_tabular["product_id"].isin(product_ids)]
predictions_tabular["abs_error"] = abs(predictions_tabular['target'] - predictions_tabular['prediction'])
total_error_tabular = np.abs(predictions_tabular['target'] - predictions_tabular['prediction']).sum() / (predictions_tabular['target'].sum())
print(f"Total Absolute Error Tabular: {total_error_tabular:.4f}")
predictions_tabular

Total Absolute Error Tabular: 0.2716


,product_id,target,prediction,abs_error
0,20001,1504.688599,1015.838257,488.850342
1,20002,1087.308594,801.011230,286.297363
2,20003,892.501282,663.387939,229.113342
3,20004,637.900024,519.842102,118.057922
4,20005,593.244446,495.175201,98.069244
...,...,...,...,...
947,21263,0.012700,0.066724,0.054024
949,21265,0.004560,0.102031,0.097471
950,21266,0.005700,0.106976,0.101276
951,21267,0.015690,0.082648,0.066958


In [16]:
submission = predictions_tabular[["product_id", "prediction"]].copy()
submission.rename(columns={"prediction": "tn"}, inplace=True)
submission.to_csv(DATA_FOLDER + "submission_autogluon_tabular.csv", index=False)
submission

,product_id,tn
0,20001,1521.456055
1,20002,1290.277954
2,20003,956.094238
3,20004,691.281189
4,20005,704.440613
...,...,...
968,21263,0.038302
970,21265,0.067041
971,21266,0.069252
972,21267,0.041744
